# Model: Incorrect Economizer Setpoint (Experimental Dataset)

## Why this fault needs two separate sub-models, not one

Per notebook 09's EDA: "setpoint too low" (6°C, 8°C) and "setpoint too high"
(12°C, 14°C) showed OPPOSITE, unexplained behaviors (suppressed vs. over-
engagement), and critically, different seasons provide valid test windows for
each direction - Winter_2022 had no valid window for the too-high severities at
all (never got cold enough for the wrong setpoint to matter), while Fall_2020 did.
Forcing one train/test split to cover both directions would either use an invalid
season for one direction or ignore the EDA's own finding. Building two separate,
honestly-scoped models instead.

## Sub-model 1: setpoint too low (6°C, 8°C)

Valid in Winter_2022 and Spring_2021 per the EDA. Using the same cross-season
design as the other two faults: train on one, test on the held-out other.

## Sub-model 2: setpoint too high (12°C, 14°C)

Valid in Fall_2020 (per notebook 09's confirmation) and, per the earlier check
in that notebook, Winter_2022's occupied-hours-only min (49.0°F) was close to
econ_12's setpoint (53.6°F) but not clearly confirmed as valid post-filtering.
Using Fall_2020 as train; a genuinely held-out cross-season test is limited here
since only Fall_2020 was confirmed valid for BOTH severities in this direction -
flagging this as a real constraint, not glossed over.

In [1]:
import sys
from pathlib import Path

ml_root = Path.cwd().parent
if str(ml_root) not in sys.path:
    sys.path.insert(0, str(ml_root))

from sklearn.ensemble import RandomForestClassifier  # noqa: E402
from sklearn.metrics import classification_report  # noqa: E402
from src.features.build_experimental_features import build_experimental_feature_table  # noqa: E402

feature_cols = ("RTU_OA_DMPR_DM", "RTU_OA_TEMP")

train_table = build_experimental_feature_table(
    baseline_path="../data/raw/experimental/ERTU_Winter_2022.csv",
    fault_paths={
        "econ_neg4_winter": "../data/raw/experimental/Inc_Eco_SP_-4_Winter_2022.csv",
        "econ_neg2_winter": "../data/raw/experimental/Inc_Eco_SP_-2_Winter_2022.csv",
    },
    feature_cols=feature_cols,
)

test_table = build_experimental_feature_table(
    baseline_path="../data/raw/experimental/ERTU_Spring_2021.csv",
    fault_paths={
        "econ_neg4_spring": "../data/raw/experimental/Inc_Eco_SP_-4_Spring_2021.csv",
        "econ_neg2_spring": "../data/raw/experimental/Inc_Eco_SP_-2_Spring_2021.csv",
    },
    feature_cols=feature_cols,
)

print(f"Train (Winter) shape: {train_table.shape}, labels:\n{train_table['label'].value_counts()}")
print(f"\nTest (Spring, held out) shape: {test_table.shape}, labels:\n{test_table['label'].value_counts()}")

Train (Winter) shape: (3600, 5), labels:
label
0    1800
1    1800
Name: count, dtype: int64

Test (Spring, held out) shape: (3600, 5), labels:
label
0    1800
1    1800
Name: count, dtype: int64


## Cross-season evaluation: train on Winter, test on held-out Spring (setpoint too
## low: 6°C, 8°C severities)

In [2]:
X_train = train_table[list(feature_cols)]
y_train = train_table["label"]
X_test = test_table[list(feature_cols)]
y_test = test_table["label"]

model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("=== Cross-season test: trained on Winter, evaluated on Spring ===")
print(classification_report(y_test, y_pred, target_names=["baseline", "econ_too_low"]))

=== Cross-season test: trained on Winter, evaluated on Spring ===
              precision    recall  f1-score   support

    baseline       0.87      0.40      0.55      1800
econ_too_low       0.61      0.94      0.74      1800

    accuracy                           0.67      3600
   macro avg       0.74      0.67      0.64      3600
weighted avg       0.74      0.67      0.64      3600



## Result: moderate cross-season performance — better than OA damper stuck's
## raw-feature collapse, worse than a clean success

Baseline recall 0.40, precision 0.87 - meaningfully better than OA damper stuck's
raw-feature result (0.11 recall) but still a real, substantial gap (missing 60% of
genuine baseline cases in the held-out season). High precision (0.87) means the
model is conservative rather than confidently wrong - when it does say "baseline,"
it's usually right, but it under-calls baseline overall.

**Plausible explanation, consistent with notebook 09's own finding**: this fault's
"too low" direction showed an unexplained OVER-engagement pattern in the original
EDA (extending well past both the fault's own and the correct setpoint) - meaning
this fault's true behavior was already known to be more complex/less predictable
than a simple threshold shift. A model trained on one season's version of that
already-unusual pattern may reasonably struggle to fully generalize to another
season's version of the same not-fully-understood behavior.

**Status**: a real, usable-but-imperfect result - meaningfully better than the
worst cases in this dataset (biased SAT sensor's near-total collapse), meaningfully
worse than the Simulated dataset's stable faults. Consistent with this fault
already being the least well-understood mechanistically in the EDA phase.

## Setpoint too high (12°C, 14°C): documented scoping limitation, not modeled

Per notebook 09's EDA, only Fall_2020 was confirmed to provide a valid test window
for both "too high" severities (Winter_2022 never got cold enough post-occupied-
hours-filtering). This means no genuine cross-season evaluation is possible for
this direction with the current dataset - any split would either misuse an invalid
season or amount to a within-season random split, which does not test what a
cross-season evaluation is meant to test, and would overstate real-world readiness
the same way random splits did throughout the Simulated-dataset work.

**Decision: not building a model for this direction.** Documenting this as a real,
honest scoping limitation rather than forcing a misleading result. A genuine model
for this fault direction would require additional data collection (a second valid
season) before it could be evaluated honestly - this is a data availability
constraint, not a modeling failure, and should be flagged to whoever owns the data-
collection roadmap rather than worked around here.

## Summary: incorrect economizer setpoint (Experimental dataset)

**Setpoint too low (6°C, 8°C)**: cross-season model built (train Winter, test
Spring). Moderate result - baseline recall 0.40, precision 0.87. Better than OA
damper stuck's raw-feature collapse, worse than a clean success. Consistent with
this fault's already-unusual, not-fully-understood EDA behavior (unexplained
over-engagement pattern).

**Setpoint too high (12°C, 14°C)**: NOT modeled. Only one season (Fall_2020) was
confirmed to provide a valid test window for this direction per the EDA - no
genuine cross-season evaluation is possible without additional data. Documented as
a real scoping/data-availability limitation, not forced into a misleading result.

**This completes all 3 Experimental-dataset fault types** (to the extent the
available data allows): OA damper stuck (partially mitigated), biased SAT sensor
(largely unresolved), incorrect economizer setpoint (partially modeled, one
direction genuinely unmodelable with current data).